In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
mcdir = f'{projdir}/snmCT/mc'

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

In [ ]:
adata = sc.read_h5ad(f'{mcdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')
adata.obs.index = [x.rsplit('-', 1)[0] for x in adata.obs.index]
adata.obs.index.duplicated().sum()

### Fig 6G: SE vs LE classifier on snm3C methylation (harmony PCs), held-out 20%

In [ ]:
def se_le_f1(idx, seed=0):
    x = adata[adata.obs.index.isin(idx)]
    y = (nuclei['ips_subtype'].reindex(x.obs.index) == 'IPS_alt').astype(int).values
    X = x.obsm['X_pca_harmony']
    train, test = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=seed)
    model = XGBClassifier(eval_metric='logloss', random_state=seed)
    model.fit(X[train], y[train])
    pred = model.predict(X[test])
    f1 = f1_score(y[test], pred, average=None)
    return len(y), f1[0], f1[1], f1.mean()

In [ ]:
m3c_all = nuclei[nuclei['assay'] == 'snm3C'].index
m3c_early = early[early['assay'] == 'snm3C'].index
rows = []
for name, idx in [('all_nuclei', m3c_all), ('early_ipsc', m3c_early)]:
    for seed in range(5):
        rows.append([name, seed, *se_le_f1(idx, seed)])
f1_df = pd.DataFrame(rows, columns=['set', 'seed', 'n', 'f1_LE', 'f1_SE', 'f1_macro'])
f1_df.groupby('set')[['n', 'f1_LE', 'f1_SE', 'f1_macro']].mean()

In [ ]:
f1_df.to_csv(f'{outdir}/se_le_mcg_classifier_f1.csv', sep='\t', index=False)

### Fig 6I: methylation UMAP of retained nuclei (as in 18_dim_reduction_just_ips_subtypes)

In [ ]:
ips_adata = adata[adata.obs.index.isin(early.index)].copy()
ips_adata.obs['ips_subtype'] = early['ips_subtype'].reindex(ips_adata.obs.index)
ips_adata.obs.groupby(['assay', 'ips_subtype']).size()

In [ ]:
sc.pp.highly_variable_genes(ips_adata, n_top_genes=20000)
sc.tl.pca(ips_adata, n_comps=50)
sc.pp.neighbors(ips_adata, random_state=0)
sc.tl.umap(ips_adata, random_state=0)
sc.pl.umap(ips_adata, color=['ips_subtype', 'line'])

In [ ]:
umap_df = pd.DataFrame(ips_adata.obsm['X_umap'], index=ips_adata.obs.index, columns=['umap0', 'umap1'])
umap_df['ips_subtype'] = ips_adata.obs['ips_subtype']
umap_df['line'] = ips_adata.obs['line']
umap_df['time'] = ips_adata.obs['time']
umap_df['assay'] = ips_adata.obs['assay']
umap_df.to_csv(f'{outdir}/ips_subtype_mcg_umap_df.csv', sep='\t')

### Fig 6J: fraction of k=50 neighbours from the same donor (as in 19_neighbors_analysis_ips_subtype)

In [ ]:
sc.pp.neighbors(ips_adata, n_neighbors=50, random_state=0, use_rep='X_pca')
dist = ips_adata.obsp['distances'].tocsr()
lines = ips_adata.obs['line'].values
props = [np.mean(lines[dist[i].indices] == lines[i]) for i in range(ips_adata.shape[0])]
ips_meta = ips_adata.obs[['line', 'assay', 'time', 'ips_subtype']].copy()
ips_meta['donor_prop'] = props
nn_summary = ips_meta.groupby(['line', 'ips_subtype'])['donor_prop'].agg(['mean', 'sem', 'size']).unstack()
nn_summary

In [ ]:
ips_meta.to_csv(f'{outdir}/ips_subtype_nn_prop.csv', sep='\t')
nn_summary.to_csv(f'{outdir}/ips_subtype_nn_prop_summary.csv', sep='\t')

### donor-balanced check: same-donor neighbours expected by chance = donor share within subtype

In [ ]:
share = ips_meta.groupby('ips_subtype')['line'].value_counts(normalize=True).unstack()
share

In [ ]:
!date